# 2. A pressure index per barri and day

**Question:** how busy is a barri on a given day, in a way that can be compared across days and argued about?

There is no agreed definition, so the index is a weighted mix of components, each ranked against the barri's own history (0-1):

| Component | Measures |
|---|---|
| `visitor_density` | non-resident visitors present / registered residents |
| `visitor_arrivals` | trips arriving in the barri made by non-residents |
| `inbound_intensity` | all trips arriving per km2 |
| `anomaly` | arrivals vs the barri's usual level for that weekday and month |
| `transit_load` | peak-hour public-transport load (see notebook 3) |

Ranking each barri against itself means every barri has its own quiet and busy days, which is what a daily discount needs. Static things (monuments, hotel beds) are not components: they don't change from day to day.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prisa import store, figures
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
con = store.connect()
from prisa import pressure

In [ ]:
pd.Series(pressure.PRESETS).apply(pd.Series).fillna(0)

In [ ]:
comps = store.read('pressure_components')
idx = store.read('pressure_index')
wide = comps.pivot_table(index=['day', 'zone_id'], columns='component', values='normalised')
wide.corr().round(2)

Components are related but not redundant, which is why the weights matter. The preset can be changed in `pressure.PRESETS` and the pipeline re-run.

In [ ]:
geom = store.read('barri_geometry')
area = store.read('zones').set_index('zone_id')['area_km2']
dens = bd = store.read('barri_day')
dens = bd.groupby('zone_id')['visitors'].mean() / area.reindex(bd['zone_id'].unique())
figures.barri_map(dens / 1e3, geom, 'Non-resident visitors present per km2 (mean day)', label='thousand / km2');